In [22]:
# Inference Pipeline: Unwetterwarnung mit Open-Meteo API

In [23]:
# Hopsworks Feature Group und Feature View laden
import os
from pathlib import Path

import hopsworks
from dotenv import load_dotenv

project_root = Path.cwd().resolve()
if not (project_root / ".env").exists():
    project_root = project_root.parent
load_dotenv(project_root / ".env")

api_key = os.getenv("HOPSWORKS_API_KEY")
project_name = os.getenv("HOPSWORKS_PROJECT_NAME")
if not api_key or not project_name:
    raise ValueError("HOPSWORKS_API_KEY und HOPSWORKS_PROJECT_NAME müssen gesetzt sein.")

project = hopsworks.login(
    api_key_value=api_key,
    project=project_name,
    host="eu-west.cloud.hopsworks.ai",
    port=443,
)
fs = project.get_feature_store()
weather_fg = fs.get_feature_group(
    name="weather_features_batch",
    version=1,
)
if weather_fg is None:
    raise RuntimeError("Die Feature Group weather_features_batch wurde nicht gefunden.")

feature_view = fs.get_feature_view(
    name="severe_weather_fv",
    version=1,
)
if feature_view is None:
    raise RuntimeError("Die Feature View severe_weather_fv wurde nicht gefunden.")

latest_batch_features = feature_view.get_batch_data(
    dataframe_type="pandas",
    primary_key=True,
)
if latest_batch_features.empty:
    raise RuntimeError("Die Feature View enthält keine gespeicherten Batch-Features.")

print(f"✅ Feature Group geladen: {weather_fg.name} (v{weather_fg.version})")
print(f"✅ Feature View geladen: {feature_view.name} (v{feature_view.version})")
print(f"📊 Gespeicherte Batch-Features geladen: {len(latest_batch_features)} Zeilen")

2026-09-16 08:40:21,998 INFO: Closing external client and cleaning up certificates.
2026-09-16 08:40:21,999 INFO: Connection closed.
2026-09-16 08:40:22,000 INFO: Initializing external client
2026-09-16 08:40:22,001 INFO: Base URL: https://eu-west.cloud.hopsworks.ai:443
2026-09-16 08:40:22,673 INFO: Python Engine initialized.



Logged in to project, explore it here https://eu-west.cloud.hopsworks.ai:443/p/44167
Finished: Reading data from Hopsworks, using Hopsworks Feature Query Service (1.01s) 
✅ Feature Group geladen: weather_features_batch (v1)
✅ Feature View geladen: severe_weather_fv (v1)
📊 Gespeicherte Batch-Features geladen: 1626 Zeilen


In [24]:
# Live-Daten abrufen (Forecast von Open-Meteo)
import requests
import pandas as pd
from datetime import datetime, timedelta


def clean_weather_data(df: pd.DataFrame) -> pd.DataFrame:
    """Bereinigt Live-Daten mit denselben Regeln wie die Feature-Pipeline."""
    df = df.copy()
    df["time"] = pd.to_datetime(df["time"], errors="coerce")
    df["location_id"] = df.apply(
        lambda row: f"{row['latitude']}_{row['longitude']}", axis=1
    )
    df = df.dropna(subset=["time", "location_id"])
    df = df.drop_duplicates(subset=["location_id", "time"], keep="last")

    valid_ranges = {
        "temperature_2m": (-90, 60),
        "relative_humidity_2m": (0, 100),
        "precipitation": (0, 1000),
        "pressure_msl": (850, 1100),
        "surface_pressure": (850, 1100),
        "cloud_cover": (0, 100),
        "wind_speed_10m": (0, 250),
        "wind_gusts_10m": (0, 350),
        "cape": (0, 10000),
    }
    numeric_columns = list(valid_ranges)
    for column, (lower, upper) in valid_ranges.items():
        values = pd.to_numeric(df[column], errors="coerce")
        df[column] = values.mask((values < lower) | (values > upper))

    df = df.sort_values(["location_id", "time"])
    for column in numeric_columns:
        df[column] = df.groupby("location_id")[column].transform(
            lambda values: values.interpolate(limit_direction="both")
        )
        df[column] = df[column].fillna(
            df.groupby("location_id")[column].transform("median")
        )

    required_columns = [
        "temperature_2m", "relative_humidity_2m", "precipitation",
        "pressure_msl", "surface_pressure", "cloud_cover",
        "wind_speed_10m", "wind_gusts_10m", "cape",
    ]
    return df.dropna(subset=required_columns).reset_index(drop=True)


def fetch_live_forecast(lat: float, lon: float, location_name: str) -> pd.DataFrame:
    """Holt aktuelle Wetterdaten und einen 3-Tage-Forecast von Open-Meteo."""
    url = "https://api.open-meteo.com/v1/forecast"
    params = {
        "latitude": lat,
        "longitude": lon,
        "hourly": [
            "temperature_2m", "relative_humidity_2m", "precipitation", "rain",
            "pressure_msl", "surface_pressure", "cloud_cover",
            "wind_speed_10m", "wind_gusts_10m", "wind_direction_10m", "cape"
        ],
        "past_days": 1,
        "forecast_days": 3,
        "timezone": "UTC"
    }

    response = requests.get(url, params=params)
    response.raise_for_status()
    data = response.json()["hourly"]

    df = pd.DataFrame(data)
    df["time"] = pd.to_datetime(df["time"])
    df["location"] = location_name
    df["latitude"] = lat
    df["longitude"] = lon
    df = clean_weather_data(df)

    print(f"✅ {len(df)} bereinigte Live-Datenpunkte für {location_name} geladen")
    return df

# Definierte Standorte
LOCATIONS = {
    "Munich": (48.1351, 11.5820),
    "Hamburg": (53.5511, 9.9937)
}

In [25]:
# User-Input für Ad-hoc-Abfrage
def get_user_location_input() -> dict:
    """
    Erlaubt Ad-hoc-Abfrage für benutzerdefinierten Standort
    """
    lat = float(input("📍 Breitengrad: "))
    lon = float(input("📍 Längengrad: "))
    name = input("🏙️ Ortsname: ")
    return {"lat": lat, "lon": lon, "name": name}

In [26]:
# Echtzeit-Feature aus der aktuellen Open-Meteo-Abfrage berechnen
from zoneinfo import ZoneInfo


def build_live_feature_vector(live_df: pd.DataFrame) -> dict:
    """Berechnet die gleichen Modell-Features wie die Feature-Pipeline."""
    live_df = live_df.copy()
    live_df["location_id"] = live_df.apply(
        lambda row: f"{row['latitude']}_{row['longitude']}", axis=1
    )
    live_df = live_df.sort_values(["location_id", "time"])
    grouped = live_df.groupby("location_id", sort=False)

    for window in [3, 6, 12]:
        live_df[f"precip_rolling_sum_{window}h"] = grouped["precipitation"].transform(
            lambda values: values.rolling(window, min_periods=1).sum()
        )
        live_df[f"wind_gust_max_{window}h"] = grouped["wind_gusts_10m"].transform(
            lambda values: values.rolling(window, min_periods=1).max()
        )
        live_df[f"pressure_mean_{window}h"] = grouped["pressure_msl"].transform(
            lambda values: values.rolling(window, min_periods=1).mean()
        )

    live_df["pressure_change_3h"] = grouped["pressure_msl"].transform(
        lambda values: values.diff(3)
    )
    live_df["pressure_drop_rate"] = live_df["pressure_change_3h"] / 3
    live_df["wind_gust_anomaly"] = grouped["wind_gusts_10m"].transform(
        lambda values: values - values.rolling(24, min_periods=1).mean()
    )
    live_df["temp_change_3h"] = grouped["temperature_2m"].transform(
        lambda values: values.diff(3)
    )

    live_df = live_df.dropna(
        subset=["pressure_drop_rate", "wind_gust_anomaly", "temp_change_3h"]
    )
    if live_df.empty:
        raise ValueError("Für die Live-Daten konnten keine vollständigen Features berechnet werden.")

    now = pd.Timestamp.now(tz="UTC").tz_localize(None)
    future_or_current = live_df[live_df["time"] >= now]
    selected = (future_or_current if not future_or_current.empty else live_df).iloc[0]
    return selected.to_dict()


def get_live_feature_vector(latitude: float, longitude: float, location_name: str) -> dict:
    """Ruft aktuelle Wetterdaten ab und erstellt daraus einen Modell-Feature-Vektor."""
    live_df = fetch_live_forecast(latitude, longitude, location_name)
    feature_vector = build_live_feature_vector(live_df)
    feature_vector["event_id"] = (
        f"{latitude}_{longitude}_{pd.Timestamp(feature_vector['time']).strftime('%Y%m%d%H%M')}"
    )
    return feature_vector


latitude, longitude = LOCATIONS["Munich"]
live_vector = get_live_feature_vector(latitude, longitude, "Munich")
print(f"⚡ Aktueller Live-Feature-Vektor: {live_vector}")

✅ 96 bereinigte Live-Datenpunkte für Munich geladen
⚡ Aktueller Live-Feature-Vektor: {'time': Timestamp('2026-09-16 09:00:00'), 'temperature_2m': 24.0, 'relative_humidity_2m': 54, 'precipitation': 0.0, 'rain': 0.0, 'pressure_msl': 1013.5, 'surface_pressure': 954.9, 'cloud_cover': 100, 'wind_speed_10m': 8.3, 'wind_gusts_10m': 22.0, 'wind_direction_10m': 236, 'cape': 100.0, 'location': 'Munich', 'latitude': 48.1351, 'longitude': 11.582, 'location_id': '48.1351_11.582', 'precip_rolling_sum_3h': 0.0, 'wind_gust_max_3h': 22.0, 'pressure_mean_3h': 1013.6999999999999, 'precip_rolling_sum_6h': 0.0, 'wind_gust_max_6h': 22.0, 'pressure_mean_6h': 1014.0833333333334, 'precip_rolling_sum_12h': 0.0, 'wind_gust_max_12h': 22.0, 'pressure_mean_12h': 1015.0833333333334, 'pressure_change_3h': -0.8999999999999773, 'pressure_drop_rate': -0.29999999999999244, 'wind_gust_anomaly': 7.216666666666667, 'temp_change_3h': 6.0, 'event_id': '48.1351_11.582_202609160900'}


In [27]:
# Modell aus der Model Registry herunterladen
import joblib
import os

mr = project.get_model_registry()
model_version = os.getenv("HOPSWORKS_MODEL_VERSION")
if model_version:
    model_meta = mr.get_model(
        name="severe_weather_classifier",
        version=int(model_version),
    )
else:
    available_models = mr.get_models(name="severe_weather_classifier")
    if not available_models:
        raise RuntimeError("Das Modell severe_weather_classifier wurde nicht gefunden.")
    model_meta = max(available_models, key=lambda candidate: int(candidate.version))

model_dir = model_meta.download()
print(f"✅ Modell heruntergeladen nach: {model_dir}")

model_path = os.path.join(model_dir, "model.joblib")
model = joblib.load(model_path)

print(f"✅ Modell geladen: {model_meta.name} (v{model_meta.version})")
print(f"📊 Trainings-Metriken: {model_meta.training_metrics}")

Downloading: 0.000%|          | 0/207801 elapsed<00:00 remaining<?

Downloading: 0.000%|          | 0/138 elapsed<00:00 remaining<?

✅ Modell heruntergeladen nach: /tmp/hopsworks/models/fhnw_p1_weather_forcasts/severe_weather_classifier/17/severe_weather_classifier_17
✅ Modell geladen: severe_weather_classifier (v17)
📊 Trainings-Metriken: {'n_test_samples': 301.0, 'n_train_samples': 1203.0, 'f1_score': 1.0, 'roc_auc': 1.0, 'positive_class_ratio': 0.07315045719035744}


In [28]:
# Real-Time Single Prediction
def run_realtime_prediction(model, feature_vector: dict, threshold: float = 0.3) -> dict:
    """
    Führt Echtzeit-Prediction für einen einzelnen Feature-Vektor durch.
    """
    expected_features = getattr(model, "feature_names_in_", None)
    if expected_features is None:
        expected_features = model.get_booster().feature_names
    if expected_features is None or len(expected_features) == 0:
        raise ValueError("Das Modell enthält keine Feature-Namen für die Inference.")

    missing_features = [
        feature_name for feature_name in expected_features
        if feature_name not in feature_vector
    ]
    if missing_features:
        raise KeyError(f"Fehlende Modell-Features: {missing_features}")

    X = pd.DataFrame([
        {feature_name: feature_vector[feature_name] for feature_name in expected_features}
    ])
    X = X.apply(pd.to_numeric, errors="coerce")

    probability = model.predict_proba(X)[0, 1]
    warning = bool(probability >= threshold)

    result = {
        "storm_probability": round(float(probability), 4),
        "storm_warning": warning,
        "risk_level": "🔴 HOCH" if probability >= 0.6 else
                       "🟠 MITTEL" if probability >= threshold else "🟢 NIEDRIG"
    }
    return result

# Ausführen
result = run_realtime_prediction(model, live_vector, threshold=0.3)
print(f"⚡ Real-Time Ergebnis: {result}")

⚡ Real-Time Ergebnis: {'storm_probability': 0.0009, 'storm_warning': False, 'risk_level': '🟢 NIEDRIG'}
